<a href="https://colab.research.google.com/github/christianlocher/llmrag/blob/main/III_Create_ChromaDB_own%20Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Package Download

In [ ]:
!python -m spacy download de_core_news_sm

# Restart Enviroment!

In [ ]:

# Necessary for Colab
#Package contains PDF library
!pip install -U chromadb pypdf langchain-community


import requests
from io import BytesIO
from langchain_community.document_loaders import PyPDFLoader



#I Text Chunking

In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

DATA_PATH = r"sample_data"
loader = PyPDFDirectoryLoader(DATA_PATH)

documents_from_url = loader.load()


#I-3 SpacyTextSplitter

In [ ]:
from langchain_text_splitters import SpacyTextSplitter
import spacy.cli
from langchain_core.documents import Document
from datetime import datetime

spacy.load('de_core_news_sm')

# be careful to use the right dictionary, see: https://spacy.io/models
text_splitter = SpacyTextSplitter(
    pipeline="de_core_news_sm",
    chunk_size=500,
    chunk_overlap=50
    )

rawtext = text_splitter.split_documents(documents_from_url)

chunks = []

for i, doc_chunk in enumerate(rawtext):
    # Create a new Document object with added metadata
    new_metadata = {
        "chunk_id": i,
        "source": "",
        "orig_source": "AnySource",
        "mandant": "OtherCompany",
        "last_updated": datetime.now().isoformat(),
        "language": "de",
        **doc_chunk.metadata # Include existing metadata from the chunk
    }
    doc = Document(
        page_content=doc_chunk.page_content, # Correctly access content from the current Document object
        metadata=new_metadata
    )
    chunks.append(doc)


i = 0

for chunk in chunks:
    print(chunk)
    print('####################################################################')

    i += 1

#III-1 Create Embeddings using Qwen3.5 0.6b and Insert into Vector Database

In this example, embeddings are created using a custom model. Also an example with an built-in model is shown.

In [5]:
##################
# Install OLLAMA #
##################

!sudo apt update
!sudo apt-get install zstd
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,479 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
52 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

In [6]:
##################
# Run     OLLAMA #
##################

import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

!ollama pull qwen3-embedding:0.6b
# Models here: https://ollama.com/library?sort=newest



In [ ]:
!pip install ollama
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

# setting the environment
# By default, Chromadb uses all-MiniLM-L6-v2 do create embeddings

CHROMA_PATH = r"chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# 1. Definiere die Ollama Embedding Funktion
# Standardmäßig läuft Ollama lokal auf Port 11434
ollama_ef = OllamaEmbeddingFunction(
    url="http://localhost:11434/api/embeddings",
    model_name="qwen3-embedding:0.6b"
)

# 2. Übergib die Embedding Funktion an die Collection
collection = chroma_client.get_or_create_collection(
    name="Furniture",
    embedding_function=ollama_ef
)


# preparing to be added in chromadb

documents = []
metadata = []
ids = []

i = 0

for chunk in chunks:
    documents.append(chunk.page_content)
    ids.append("ID"+str(i))
    metadata.append(chunk.metadata)
    print(chunk.page_content)
    print('############################')

    i += 1

# adding to chromadb


collection.upsert(
    documents=documents,
    metadatas=metadata,
    ids=ids
)

In [ ]:
while True:

  user_query = input("How can I help you?")

  if user_query.lower() == "quit":
            break

  results = collection.query(
      query_texts=[user_query],
      n_results=10

  )

  print(results['documents'])
  print(results['metadatas'])

